# Week 5 - Module 5: Generative Anomaly Detection
### BiLSTM-VAE: reconstruction-error anomaly detection + synthetic rare-event generation

Reuses `vitals_data.py` (same loader as Module 4). The BiLSTM here is BOTH the VAE encoder AND decoder. **Trained only on normal-looking sequences** (`labels == 0`) - training on a mix would teach the model to reconstruct abnormal patterns too, defeating the anomaly signal.

Run all cells top to bottom.

In [ ]:
import os
import sys
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Cardiac_Risk_Prediction'
except ImportError:
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.append(f'{PROJECT_DIR}/src')

print(f'Working directory permanently mapped to: {PROJECT_DIR}')


In [ ]:
import sys
sys.path.append(f'{PROJECT_DIR}/src')

from vitals_data import load_vitals_sequences, SEQ_LEN, FORECAST_HORIZON
from vae_models import VitalsBiLSTMVAE, vae_loss

## Load vitals sequences and filter to normal-only for training
**Real-MIMIC data currently has no deterioration label wired up** (see `vitals_data.py` docstring - would need SOFA/ICD-based labeling on top of raw vitals). Use the synthetic fallback here (`mimic_dir=None`) so the normal/abnormal filter below actually has labels to work with; swap in real MIMIC data once that labeling extraction exists.

In [ ]:
X, y, labels = load_vitals_sequences(mimic_dir=None, n_samples_fallback=1000)
print(f"Total sequences: {len(X)} | Normal: {int((labels==0).sum())} | Deteriorating: {int((labels==1).sum())}")

# Module 5 requirement: train ONLY on normal sequences
normal_mask = labels == 0
X_normal, y_normal = X[normal_mask], y[normal_mask]
# Keep the abnormal set aside - used below to sanity-check the anomaly detector
X_abnormal, y_abnormal = X[~normal_mask], y[~normal_mask]
print(f"Training on {len(X_normal)} normal sequences; holding out {len(X_abnormal)} abnormal ones for evaluation.")

## Split, standardize, and build DataLoaders (normal sequences only)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

X_train, X_val, y_train, y_val = train_test_split(X_normal, y_normal, test_size=0.2, random_state=42)

n_features = X_train.shape[-1]
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, n_features))

def scale(arr):
    shape = arr.shape
    return scaler.transform(arr.reshape(-1, n_features)).reshape(shape)

X_train_s, X_val_s = scale(X_train), scale(X_val)
y_train_s, y_val_s = scale(y_train), scale(y_val)
X_abnormal_s, y_abnormal_s = scale(X_abnormal), scale(y_abnormal)

def make_target(x, y):
    return np.concatenate([x, y], axis=1)  # window + horizon = decoder's reconstruction target

train_ds = TensorDataset(torch.tensor(X_train_s, dtype=torch.float32),
                          torch.tensor(make_target(X_train_s, y_train_s), dtype=torch.float32))
val_ds = TensorDataset(torch.tensor(X_val_s, dtype=torch.float32),
                        torch.tensor(make_target(X_val_s, y_val_s), dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=32)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

## Build, train (or load checkpoint) VitalsBiLSTMVAE

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

CHECKPOINT_PATH = f'{PROJECT_DIR}/checkpoints/module5_vae.pt'

model = VitalsBiLSTMVAE(n_features=n_features, seq_len=SEQ_LEN, horizon=FORECAST_HORIZON).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

if os.path.exists(CHECKPOINT_PATH):
    print("Found pre-trained checkpoint in Drive! Loading instead of retraining.")
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
else:
    print("No checkpoint found - training from scratch.")
    EPOCHS = 60
    train_losses, val_losses = [], []
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        for batch_x, batch_target in train_loader:
            batch_x, batch_target = batch_x.to(device), batch_target.to(device)
            optimizer.zero_grad()
            recon, mu, log_var = model(batch_x)
            loss, recon_loss, kl_loss = vae_loss(recon, batch_target, mu, log_var)
            loss.backward()
            optimizer.step()
            running += loss.item() * batch_x.size(0)
        train_losses.append(running / len(train_ds))

        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for batch_x, batch_target in val_loader:
                batch_x, batch_target = batch_x.to(device), batch_target.to(device)
                recon, mu, log_var = model(batch_x)
                v_loss, _, _ = vae_loss(recon, batch_target, mu, log_var)
                val_running += v_loss.item() * batch_x.size(0)
        val_losses.append(val_running / len(val_ds))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} - train loss: {train_losses[-1]:.4f} - val loss: {val_losses[-1]:.4f}")

    torch.save(model.state_dict(), CHECKPOINT_PATH)
    print(f"Saved trained model to {CHECKPOINT_PATH}")

    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label='train'); plt.plot(val_losses, label='val')
    plt.title('Module 5 - BiLSTM-VAE Loss (normal sequences only)')
    plt.xlabel('Epoch'); plt.ylabel('Total loss (recon + KL)'); plt.legend()
    plt.savefig(f'{PROJECT_DIR}/outputs/week5_vae_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

## Anomaly detection: does reconstruction error actually separate normal vs abnormal?

In [ ]:
normal_target = torch.tensor(make_target(X_val_s, y_val_s), dtype=torch.float32).to(device)
normal_input = torch.tensor(X_val_s, dtype=torch.float32).to(device)
normal_scores = model.reconstruction_error(normal_input, normal_target).cpu().numpy()

abnormal_target = torch.tensor(make_target(X_abnormal_s, y_abnormal_s), dtype=torch.float32).to(device)
abnormal_input = torch.tensor(X_abnormal_s, dtype=torch.float32).to(device)
abnormal_scores = model.reconstruction_error(abnormal_input, abnormal_target).cpu().numpy()

threshold = float(np.percentile(normal_scores, 95))
print(f"95th-percentile threshold (fit on normal val data): {threshold:.4f}")
print(f"Normal mean error:   {normal_scores.mean():.4f}")
print(f"Abnormal mean error: {abnormal_scores.mean():.4f}")

if len(abnormal_scores):
    detected = (abnormal_scores > threshold).mean()
    print(f"Detection rate on held-out abnormal sequences: {detected*100:.1f}%")

plt.figure(figsize=(7, 4))
plt.hist(normal_scores, bins=30, alpha=0.6, label='normal')
if len(abnormal_scores):
    plt.hist(abnormal_scores, bins=30, alpha=0.6, label='abnormal (deterioration)')
plt.axvline(threshold, color='red', linestyle='--', label='95th pct threshold')
plt.title('Reconstruction Error: Normal vs Abnormal Sequences')
plt.xlabel('Reconstruction error'); plt.legend()
plt.savefig(f'{PROJECT_DIR}/outputs/week5_anomaly_separation.png', dpi=150, bbox_inches='tight')
plt.show()

## Synthetic vitals generation (for Module 1/3 rare-event augmentation)

In [ ]:
synthetic = model.generate(n_samples=5, device=device).cpu().numpy()
synthetic_real = scaler.inverse_transform(
    synthetic.reshape(-1, n_features)).reshape(synthetic.shape)

fig, axes = plt.subplots(1, 5, figsize=(20, 3), sharey=True)
for i, ax in enumerate(axes):
    ax.plot(synthetic_real[i, :, 0], label='HR')
    ax.plot(synthetic_real[i, :, 1], label='SpO2')
    ax.set_title(f'Synthetic sample {i+1}')
axes[0].legend()
plt.savefig(f'{PROJECT_DIR}/outputs/week5_synthetic_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## Notes for the progress report
- Trained **only on normal (non-deteriorating) synthetic sequences** - this is the correct setup for a valid anomaly detector; report the normal-vs-abnormal mean error separation and detection rate above honestly, including if it's weak (that's a genuine finding worth stating, not something to hide).
- The `mu` (encoder output) returned by `model(x, return_features=True)` is what Week 6 fusion will consume from this branch.
- Once real MIMIC deterioration/sepsis labels are wired into `vitals_data.py`, re-run this notebook on real data for the final report - currently synthetic-only, stated here explicitly.